In [ ]:
# Single-asset GLD VaR / ES (5y TradingView CSV)
import pandas as pd
import numpy as np
import scipy as sp
from scipy import optimize
from pathlib import Path
import matplotlib.pyplot as plt
%matplotlib inline

import market_risk as mr
from market_risk import viz

Path("images").mkdir(exist_ok=True)
ALPHA = mr.ALPHA


In [ ]:
data = pd.read_csv("data/GLD_5yrs_historical_prices.csv", parse_dates=["Date"]).sort_values("Date")
dates = data[["Date"]].to_numpy()
prices = data[["Close/Last"]].to_numpy()
rets = (prices[1:] - prices[:-1]) / prices[:-1]
logrets = np.log(1 + rets)


In [ ]:
fig = viz.gld_returns_figure(dates, prices, rets, logrets)
plt.tight_layout()
fig.savefig("images/risk-1-GLD-returns.png", bbox_inches="tight")


In [ ]:
mu, sigma = sp.stats.norm.fit(logrets)
df, loc, scale = sp.stats.t.fit(logrets)
kde = sp.stats.gaussian_kde(logrets.T)
def kde_q(q):
    return optimize.brentq(lambda x: kde.integrate_box_1d(-np.inf, x) - q, logrets.min(), logrets.max())


In [ ]:
_norm_var_log, _norm_es_log = mr.parametric_var_es_normal(logrets.flatten(), ALPHA)
_t_var_log, _t_es_log = mr.parametric_var_es_t(logrets.flatten(), ALPHA)
_hist_var_log = kde_q(ALPHA)
tail = logrets.flatten()[logrets.flatten() <= _hist_var_log]
_hist_es_log = float(tail.mean()) if len(tail) else _hist_var_log
print("VaR%:", mr.log_var_to_loss(_norm_var_log)*100, mr.log_var_to_loss(_t_var_log)*100, mr.log_var_to_loss(_hist_var_log)*100)


In [ ]:
rows = []
for name, v, e in [("normal", _norm_var_log, _norm_es_log), ("t", _t_var_log, _t_es_log), ("historical", _hist_var_log, _hist_es_log)]:
    breach = logrets.flatten() <= v
    rows.append({
        "method": name,
        "kupiec_p": mr.kupiec_test(int(breach.sum()), len(logrets))["p_value"],
        "cc_p": mr.christoffersen_conditional_coverage(breach)["p_value"],
        "basel_green": mr.basel_traffic_light(breach)["green_pct"],
    })
pd.DataFrame(rows)
